# 02 Crazyflie — Changing UWB Measurement Noise

Simulates a Crazyflie flight where **UWB** sensor noise changes in three phases.
ToF receives constant baseline noise throughout.

The `noise_density` and `random_walk` parameters of the UWB noise profile are scaled.

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

from core.sensor_stream import SensorStream, SensorOrigin
from pipeline.noise import NoiseInjector
from pipeline.noise.core.noise_profile import NoiseProfile
from pipeline.synthetic import SyntheticSensorGenerator, SyntheticTimeOfFlightDistance, SyntheticUWBTransceiver
from rotorpy_simulation.crazyflie import Crazyflie, CrazyflieIMU
from rotorpy_simulation.crazyflie_simulation import CrazyflieSimulation

In [ ]:
SIMULATION_NAME: str = "02_crazyflie_changing_measurement_noise"
DATA_PATH: str = "../../simulated_data/rotorpy/noise/" + SIMULATION_NAME
plt.style.use('default')

NOISE_FACTOR_NORMAL = 1.0
NOISE_FACTOR_HIGH = 2.0
NOISE_FACTOR_LOW = 0.5

## 1. Object Generation & Run

In [ ]:
from rotorpy_simulation.crazyflie.trajectories import get_circular_trajectory

crazyflie = Crazyflie()
imu_sensor = CrazyflieIMU()
trajectory = get_circular_trajectory()

simulation = CrazyflieSimulation(
    crazyflie_builder=crazyflie,
    imu_sensor_builder=imu_sensor,
    trajectory=trajectory,
)
simulation.simulation_duration = 60.0

result = simulation.run(SIMULATION_NAME)

## 2. Validation
Sanity-check the trajectory and ground-truth data.

In [ ]:
gt = result.ground_truth
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot(gt.position[:, 0], gt.position[:, 1], gt.position[:, 2], color='royalblue', linewidth=1)
ax.scatter(*gt.position[0], color='green', s=80, marker='o', label='Start')
ax.scatter(*gt.position[-1], color='red', s=80, marker='x', label='End')
ax.set_xlabel('East [m]')
ax.set_ylabel('North [m]')
ax.set_zlabel('Up [m]')
ax.set_title('Crazyflie Ground-Truth Trajectory')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
result.ground_truth.to_dataframe().head(10)

## 3. Synthetic Sensor Generation
Generate ToF with constant baseline noise. UWB streams are generated clean; noise splicing is applied in section 4.

In [ ]:
tof_noise_profile = NoiseProfile.from_yaml("../../pipeline/noise/profiles/VL53L1x_ToF-Distance.yaml")
uwb_noise_profile = NoiseProfile.from_yaml("../../pipeline/noise/profiles/DWM1000_UWB_Transceiver.yaml")

_anchor_positions = [
    np.array([2.32, 1.39, 2.08]),
    np.array([2.32, -1.43, 2.08]),
    np.array([2.34, 1.39, 0.23]),
    np.array([2.32, -1.43, 0.20]),
    np.array([-2.32, 1.39, 2.25]),
    np.array([-2.32, -1.43, 1.87]),
    np.array([-2.32, 1.39, 0.23]),
    np.array([-2.32, -1.43, 0.24]),
]

synthetic_sensors = [
                        (SyntheticTimeOfFlightDistance(name="time_of_flight_distance"),
                         NoiseInjector(tof_noise_profile, seed=10), None),
                    ] + [
                        (SyntheticUWBTransceiver(name=f"uwb_transceiver_{i}", anchor_position=pos, anchor_id=i), None,
                         None)
                        for i, pos in enumerate(_anchor_positions)
                    ]

generator = SyntheticSensorGenerator(sensors=synthetic_sensors)
generator.apply(result)

print(f"Clean streams: {list(result.sensors_clean.keys())}")
print(f"Noisy streams: {list(result.sensors_noisy.keys())}")

## 4. UWB Noise Variation Splicing

The flight is split into three equal time windows and a different `NoiseInjector` (scaled UWB profile)
is applied to each window; the results are concatenated into a single `SensorStream`.

In [ ]:
def scale_profile(profile: NoiseProfile, factor: float) -> NoiseProfile:
    return NoiseProfile(
        name=f"{profile.name}_x{factor}",
        unit=profile.unit,
        noise_density=profile.noise_density * factor,
        bias_fixed=profile.bias_fixed,
        random_walk=profile.random_walk * factor,
        scale_factor_error=profile.scale_factor_error,
        range_min=profile.range_min,
        range_max=profile.range_max,
        resolution=profile.resolution,
    )


def splice_noisy_stream(
        clean_stream: SensorStream,
        phase_boundaries: list,
        injectors: list,
        noisy_name: str,
) -> SensorStream:
    assert len(injectors) == len(phase_boundaries) - 1

    parts_time = []
    parts_data = []

    for inj, t_start, t_end in zip(injectors, phase_boundaries[:-1], phase_boundaries[1:]):
        mask = (clean_stream.time >= t_start) & (clean_stream.time < t_end)
        if not np.any(mask):
            continue

        segment = SensorStream(
            name=clean_stream.name,
            time=clean_stream.time[mask].copy(),
            data=clean_stream.data[mask].copy(),
            channels=list(clean_stream.channels),
            unit=clean_stream.unit,
            sampling_rate=clean_stream.sampling_rate,
            origin=SensorOrigin.SYNTHETIC,
        )
        noisy_segment = inj.apply(segment)
        parts_time.append(noisy_segment.time)
        parts_data.append(noisy_segment.data)

    return SensorStream(
        name=noisy_name,
        time=np.concatenate(parts_time),
        data=np.concatenate(parts_data, axis=0),
        channels=list(clean_stream.channels),
        unit=clean_stream.unit,
        sampling_rate=clean_stream.sampling_rate,
        origin=SensorOrigin.PIPELINE_PROCESSED,
    )

In [ ]:
t_total = result.ground_truth.time[-1]
t1 = t_total / 3.0
t2 = 2.0 * t_total / 3.0
phase_boundaries = [0.0, t1, t2, t_total + 1e-6]

print(f"Total duration : {t_total:.1f} s")
print(f"Phase 1 (normal ×{NOISE_FACTOR_NORMAL}):   0 – {t1:.1f} s")
print(f"Phase 2 (high   ×{NOISE_FACTOR_HIGH}):  {t1:.1f} – {t2:.1f} s")
print(f"Phase 3 (low    ×{NOISE_FACTOR_LOW}):  {t2:.1f} – {t_total:.1f} s")

for i in range(8):
    uwb_clean = result.sensors_clean[f"uwb_transceiver_{i}_clean"]
    base_seed = 20 + i * 10
    uwb_injectors = [
        NoiseInjector(scale_profile(uwb_noise_profile, NOISE_FACTOR_NORMAL), seed=base_seed, channels=[0]),
        NoiseInjector(scale_profile(uwb_noise_profile, NOISE_FACTOR_HIGH), seed=base_seed + 1, channels=[0]),
        NoiseInjector(scale_profile(uwb_noise_profile, NOISE_FACTOR_LOW), seed=base_seed + 2, channels=[0]),
    ]
    uwb_noisy = splice_noisy_stream(uwb_clean, phase_boundaries, uwb_injectors, f"uwb_transceiver_{i}")
    result.add_sensor_noisy(f"uwb_transceiver_{i}", uwb_noisy)

print(f"\nNoisy sensor streams: {list(result.sensors_noisy.keys())}")

## 5. Visualization
Verify the noise variation is visible in the sensor streams.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

phase_labels = [
    (t1 / 2, f"Normal ×{NOISE_FACTOR_NORMAL}"),
    ((t1 + t2) / 2, f"High ×{NOISE_FACTOR_HIGH}"),
    ((t2 + t_total) / 2, f"Low ×{NOISE_FACTOR_LOW}"),
]

uwb_c = result.sensors_clean["uwb_transceiver_0_clean"]
uwb_n = result.sensors_noisy["uwb_transceiver_0"]
ax.plot(uwb_c.time, uwb_c.data[:, 0], color='green', linewidth=1, label="Clean", zorder=3)
ax.plot(uwb_n.time, uwb_n.data[:, 0], color='red', linewidth=0.7, alpha=0.7, label="Noisy")
ax.axvline(t1, color='darkorange', linestyle='--', linewidth=1.5)
ax.axvline(t2, color='darkorange', linestyle='--', linewidth=1.5)
for x, label in phase_labels:
    ax.text(x, ax.get_ylim()[0] + 0.97 * (ax.get_ylim()[1] - ax.get_ylim()[0]),
            label, ha='center', va='top', fontsize=9, color='dimgray',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))
ax.set_xlabel("Time [s]")
ax.set_ylabel(f"Distance [{uwb_c.unit}]")
ax.set_title("UWB Transceiver 0 — changing measurement noise")
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
uwb_c = result.sensors_clean["uwb_transceiver_0_clean"]
uwb_n = result.sensors_noisy["uwb_transceiver_0"]
residual = uwb_n.data[:, 0] - uwb_c.data[:, 0]

masks = {
    f"Phase 1 normal (×{NOISE_FACTOR_NORMAL})": uwb_c.time < t1,
    f"Phase 2 high   (×{NOISE_FACTOR_HIGH})": (uwb_c.time >= t1) & (uwb_c.time < t2),
    f"Phase 3 low    (×{NOISE_FACTOR_LOW})": uwb_c.time >= t2,
}

print("UWB transceiver 0 residual statistics per phase:")
print(f"{'Phase':<35} {'std [m]':>10} {'mean [m]':>10}")
print("-" * 57)
for label, mask in masks.items():
    r = residual[mask]
    print(f"{label:<35} {r.std():>10.4f} {r.mean():>10.4f}")

## 6. Export
Write CSV files and a pickle that is automatically discovered by `99_crazyflie_multi_evaluation.ipynb`.

In [ ]:
result.export_csv_data(DATA_PATH)
result.save(DATA_PATH + "/" + SIMULATION_NAME + ".pkl")